How an LLM Gets \(\pi(y|x)\)

This is the implementation-critical part of DPO.

We keep saying:

$$ \pi_\theta(y|x) $$

but an LLM doesn't directly output "probability of this entire answer."

It predicts one token at a time.

1. Start with a simple example

Prompt:

What is 2 + 2?

Chosen response:

It is 4.

Tokenizer might turn it into:

"It" → token 1
"is" → token 2
"4"  → token 3
"."  → token 4

The LLM predicts:

$$ P(\text{It}|x) $$

then:

$$ P(\text{is}|x,\text{It}) $$

then:

$$ P(4|x,\text{It},\text{is}) $$

then:

$$ P(.|x,\text{It},\text{is},4) $$
2. How do we get the probability of the whole answer?

Probability of a sequence is the product of the conditional probabilities:

$$ \boxed{ \pi(y|x) = \prod_{t=1}^{T} P(y_t|x,y_{<t}) } $$

So if the model gives:

$$ P(\text{It})=0.8 $$ $$ P(\text{is})=0.9 $$ $$ P(4)=0.95 $$ $$ P(.)=0.9 $$

then:

$$ \pi(y|x) = 0.8\times0.9\times0.95\times0.9 $$ $$ =0.6156 $$

So the model assigns roughly 61.6% probability to that exact sequence.

3. But we don't actually multiply these

Because sequences can contain hundreds/thousands of tokens.

Multiplying lots of numbers smaller than 1 causes numerical underflow.

Instead we use logs:

$$ \log\pi(y|x) = \sum_t \log P(y_t|x,y_{<t}) $$

For our example:

$$ \log\pi(y|x) = \log0.8+\log0.9+\log0.95+\log0.9 $$

This is what we'll actually calculate in code.

4. Where do these token probabilities come from?

The Transformer produces logits.

For example, at one position:

vocabulary
   ↓
logits
   ↓
softmax
   ↓
probabilities

Suppose:

$$ z=[2.0,1.0,0.1] $$

Softmax:

$$ P_i= \frac{e^{z_i}} {\sum_j e^{z_j}} $$

giving something like:

token A → 0.659
token B → 0.242
token C → 0.099

The probability of the actual next token is the one we select from this distribution.

5. But there's a subtle implementation detail

Suppose the sequence is:

The cat sat

When predicting cat, the model sees:

The

When predicting sat, it sees:

The cat

So the model's logits are shifted relative to the target tokens.

Conceptually:

Input:     The    cat
Target:    cat    sat

That's the same causal-language-model setup used during normal cross-entropy training.

For DPO, we use that mechanism to calculate the likelihood of the already-given response, rather than generating it.

6. This is important: DPO does NOT need to generate the answer

This is one of the biggest differences from PPO RLHF.

We already have:

prompt
chosen response
rejected response

DPO simply asks the model:

"How probable is this existing response?"

So:

chosen ─────→ πθ → log P(chosen)
rejected ───→ πθ → log P(rejected)

and the same thing through the frozen reference model:

chosen ─────→ πref → log P(chosen)
rejected ───→ πref → log P(rejected)

No sampling is required.

7. We only score the response, not the prompt

Suppose:

Prompt:
Explain gravity.

Response:
Gravity is an attractive force...

We don't want:

$$ \log P(\text{Explain}) + \log P(\text{gravity}) +\cdots $$

included in the DPO score.

We want:

$$ \boxed{ \log P(\text{response}|\text{prompt}) } $$

So during implementation we create a loss mask:

Prompt tokens:    0 0 0 0 0
Response tokens:  1 1 1 1 1

Only tokens marked 1 contribute.

8. Why does this matter?

Because otherwise we'd be measuring:

"How likely is the entire prompt + answer sequence?"

instead of:

"How likely is this answer given this prompt?"

DPO cares about the second one.

9. Now connect it back to the DPO equation

Remember:

$$ \log \frac{\pi_\theta(y|x)} {\pi_{\rm ref}(y|x)} $$

Using the log identity:

$$ \log\frac{a}{b} = \log a-\log b $$

we calculate:

$$ \boxed{ \log\pi_\theta(y|x) - \log\pi_{\rm ref}(y|x) } $$

So for the chosen answer:

$$ \Delta_w= \log\pi_\theta(y_w|x) - \log\pi_{\rm ref}(y_w|x) $$

For rejected:

$$ \Delta_l= \log\pi_\theta(y_l|x) - \log\pi_{\rm ref}(y_l|x) $$

Then DPO uses:

$$ \boxed{ \Delta_w-\Delta_l } $$

and feeds it into the sigmoid.

10. The complete data flow

This is what we eventually implement:

             prompt
                │
       ┌────────┴────────┐
       ↓                 ↓
   chosen              rejected
       │                 │
       ↓                 ↓
     πθ model          πθ model
       │                 │
       ↓                 ↓
 log P(chosen)      log P(rejected)
       │                 │
       └────────┬────────┘
                │
         compare with
                │
       frozen πref model
                │
                ↓
     relative log probabilities
                │
                ↓
          DPO objective
                │
                ↓
          update πθ
The key thing to lock in

When you see:

$$ \pi_\theta(y|x) $$

don't imagine some mysterious single probability coming out of the Transformer.

Think:

$$ \boxed{ \pi_\theta(y|x) = \prod_t P_\theta(y_t|x,y_{<t}) } $$

and in practice:

$$ \boxed{ \log\pi_\theta(y|x) = \sum_t\log P_\theta(y_t|x,y_{<t}) } $$

That is the bridge between the DPO mathematics and actual Transformer tensors.

For a given already-tokenized sequence, getting all the logits is one forward pass, not one forward pass per token.

Example:

Input:
"The cat sat"

The Transformer processes the whole sequence in parallel:

"The"   "cat"   "sat"
  ↓       ↓       ↓
logits₁ logits₂ logits₃

Each position's logits predict the next token:

"The"        → predicts "cat"
"The cat"    → predicts "sat"
"The cat sat"→ predicts next token

So for DPO, since the response already exists, we do:

$$ \boxed{\text{one forward pass} \rightarrow \text{logits for every position}} $$

Then we simply pick out the probability of the actual response token at each position.